# Khảo Sát Tập Dữ Liệu Phát Hiện Đối Tượng COCO (Phase 0.3 COCO Detection Explore)

Nghiên cứu này tiến hành phân tích cục bộ đối với 16 phân lớp đối tượng (object classes) phục vụ tính năng hỗ trợ tiếp cận (accessibility) trong tập dữ liệu COCO 2017. Tập lệnh được thiết kế với cơ chế kiểm soát an toàn (fail-safe); trong trường hợp dữ liệu chưa được nạp (downloaded) tại thư mục `data/` bị vô hiệu hóa bởi git, hệ thống chỉ xuất báo cáo rỗng thay vì gây lỗi gián đoạn tiến trình (execution exception).

In [ ]:
import json
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from src.training.scripts.data_utils import load_yaml

# Thiết lập môi trường đồ họa và đường dẫn cơ sở
sns.set_theme(style="whitegrid")
plt.rcParams.update({"figure.figsize": (10, 6), "axes.titlesize": 14})

repo = Path("..").resolve()
config = load_yaml(repo / "configs/datasets/object_detection_accessibility.yaml")
classes_path = repo / config["output"]["canonical_names_path"]
yolo_root = repo / config["output"]["yolo_root"]
base_dataset_root = repo / config["base_dataset"]["root"]

# Đọc danh sách các phân lớp ưu tiên
if classes_path.exists():
    target_classes = [
        line.strip()
        for line in classes_path.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]
else:
    target_classes = config["classes"]["public_bootstrap"]

## 1. Trích Xuất Số Liệu Tập Dữ Liệu (Dataset Counts Extraction)
Phân mục dưới đây thực hiện tải cấu trúc tệp JSON của COCO. Trong trường hợp dữ liệu hiện diện, mã lệnh sẽ đọc thông tin tập chú thích (annotations) nhằm thống kê số lượng hình ảnh và số hộp giới hạn (bounding boxes) khớp với 16 lớp đối tượng yêu cầu.

In [ ]:
def analyze_coco_split(split_name, json_path):
    if not json_path.exists():
        return None
    print(f"Đang phân tích cú pháp tệp {split_name} từ {json_path.name}...")
    with open(json_path, encoding="utf-8") as f:
        data = json.load(f)

    # Bản đồ id -> tên danh mục
    cat_id_to_name = {cat["id"]: cat["name"] for cat in data["categories"]}
    target_cat_ids = {
        cat["id"] for cat in data["categories"] if cat["name"] in target_classes
    }

    images_with_target = set()
    annotations_count = 0
    class_distribution = Counter()
    bboxes = []

    for ann in data["annotations"]:
        if ann["category_id"] in target_cat_ids:
            images_with_target.add(ann["image_id"])
            annotations_count += 1
            class_name = cat_id_to_name[ann["category_id"]]
            class_distribution[class_name] += 1

            # bbox format: [x, y, width, height]
            x, y, w, h = ann["bbox"]
            area = w * h
            bboxes.append(
                {
                    "split": split_name,
                    "class": class_name,
                    "width": w,
                    "height": h,
                    "area": area,
                }
            )

    return {
        "total_images": len(data["images"]),
        "target_images": len(images_with_target),
        "target_annotations": annotations_count,
        "class_dist": class_distribution,
        "bboxes": bboxes,
    }


# Mô phỏng đọc hai tập train và val
train_json = base_dataset_root / "annotations" / "instances_train2017.json"
val_json = base_dataset_root / "annotations" / "instances_val2017.json"

train_stats = analyze_coco_split("train2017", train_json)
val_stats = analyze_coco_split("val2017", val_json)

counts_data = []
if train_stats:
    counts_data.append(
        {
            "Split": "train2017",
            "Total Images": train_stats["total_images"],
            "Images w/ targets": train_stats["target_images"],
            "Annotations API": train_stats["target_annotations"],
        }
    )
if val_stats:
    counts_data.append(
        {
            "Split": "val2017",
            "Total Images": val_stats["total_images"],
            "Images w/ targets": val_stats["target_images"],
            "Annotations API": val_stats["target_annotations"],
        }
    )

if counts_data:
    df_counts = pd.DataFrame(counts_data)
    display(df_counts)
else:
    print("Dữ liệu chưa khả dụng tại data/external/coco2017/annotations.")

## 2. Phân Phối Cấp Lớp (Class Distribution)
Đoạn mã hiện tại tiến hành tổng hợp giá trị tần suất xuất hiện (frequency values) của mỗi phân lớp đối tượng. Biểu đồ cột (Bar chart) sẽ cung cấp một phương tiện thị giác hóa toàn vẹn nhằm phát hiện sự bất đối xứng dữ liệu (Class Imbalance) nếu tồn tại.

In [ ]:
dist_data = []
for cls in target_classes:
    t_count = train_stats["class_dist"].get(cls, 0) if train_stats else 0
    v_count = val_stats["class_dist"].get(cls, 0) if val_stats else 0
    dist_data.append(
        {
            "Class": cls,
            "Train Annotations": t_count,
            "Val Annotations": v_count,
            "Total": t_count + v_count,
        }
    )

df_dist = pd.DataFrame(dist_data).sort_values(by="Total", ascending=False)
if not df_dist["Total"].sum() == 0:
    display(df_dist)

    plt.figure(figsize=(14, 7))
    sns.barplot(data=df_dist, x="Class", y="Total", color="steelblue")
    plt.xticks(rotation=45, ha="right")
    plt.title(
        "Phân Đối Tần Suất Lớp (Class Distribution) Cho Các Hạng Mục Hỗ Trợ Tiếp Cận"
    )
    plt.ylabel("Tổng Số Lượng Nhãn (Annotations)")
    plt.tight_layout()
    plt.show()
else:
    print("Không tìm thấy tần suất để vẽ biểu đồ Class Distribution.")

## 3. Khảo Sát Đặc Trưng Hình Học Hộp Giới Hạn (Bounding Box Geometric Summary)
Việc đo lường các đặc trưng biến thiên của một Hộp giới hạn (Bounding box) như Chiều rộng (Width), Chiều cao (Height) và Diện tích (Area) có tính chất thiết yếu cho quá trình ấn định mỏ neo (Anchor box optimization) tại mô hình YOLO.
Đoạn mã tiến hành trích xuất số liệu trung vị (mean), phân vị thứ 50 (p50) và 95 (p95) để lượng hóa sự biến thiên này và sử dụng biểu đồ nến (Boxplot) để hiển thị phổ dữ liệu.

In [ ]:
all_bboxes = []
if train_stats:
    all_bboxes.extend(train_stats["bboxes"])
if val_stats:
    all_bboxes.extend(val_stats["bboxes"])

if all_bboxes:
    df_bbox = pd.DataFrame(all_bboxes)

    # Tính toán bảng phần trăm
    summary = []
    for split in df_bbox["split"].unique():
        sub_df = df_bbox[df_bbox["split"] == split]
        summary.append(
            {
                "Split": split,
                "W_mean": sub_df["width"].mean(),
                "W_p50": sub_df["width"].median(),
                "W_p95": np.percentile(sub_df["width"], 95),
                "H_mean": sub_df["height"].mean(),
                "H_p50": sub_df["height"].median(),
                "H_p95": np.percentile(sub_df["height"], 95),
                "Area_mean": sub_df["area"].mean(),
                "Area_p50": sub_df["area"].median(),
                "Area_p95": np.percentile(sub_df["area"], 95),
            }
        )
    display(pd.DataFrame(summary).round(2))

    # Boxplot trực quan hóa dị biệt diện tích Hộp giới hạn
    plt.figure(figsize=(10, 5))
    sns.boxplot(
        data=df_bbox, x="split", y="area", showfliers=False
    )  # Bỏ qua ngoại lai quá lớn để biểu đồ dễ đọc
    plt.title("Phân Phối Biến Thiên Diện Tích Hộp Giới Hạn (Bbox Area Variance)")
    plt.ylabel("Diện tích (Area in pixels)")
    plt.xlabel("Phân vùng Dữ liệu (Split)")
    plt.show()
else:
    print("Dữ liệu Bounding Box hiện tại trống hoặc chưa được xử lý.")

## 4. Đánh Giá Trạng Thái Validation Định Dạng YOLO
Xác nhận lại tính hợp lệ đối với cấu trúc sao chụp (conversion integrity) của thư mục `data/processed/object_detection_accessibility_merged`.

In [ ]:
if target_classes and yolo_root.exists():
    image_count = sum(1 for _ in (yolo_root / "images").rglob("*.*"))
    label_count = sum(1 for _ in (yolo_root / "labels").rglob("*.txt"))
    display(
        {
            "YOLO image files": image_count,
            "YOLO label files": label_count,
            "Full validation": "Run CLI validate outside notebook",
        }
    )
else:
    print("Thư mục Processed YOLO chưa tồn tại hoặc chưa được khởi tạo.")